# IRIS: 04 Metro Manila outlook

This notebook looks at what the available data can say about the weeks and months ahead for Metro Manila.

1. **Monthly odds.** From 45 years of CHIRPS 10-day rainfall for NCR (1981 to 2025), the chance that each calendar
   month has at least one 10-day period above the leptospirosis alert threshold (NCR's 90th percentile, 166 mm),
   and above a flood level.
2. **Flood level.** The 10-day rainfall of Metro Manila's major recent floods (Ondoy 2009, Habagat 2012,
   Maring 2013, Habagat 2018, Carina 2024) sets a rain level above which wide flooding is likely. This is a rain
   trigger, not a hydrological flood model.
3. **Inputs for the live forecast.** One point per province, used to fetch a 16-day rainfall forecast from
   Open-Meteo in the app, and a baseline of weekly national cases with no extreme rain (the forecast model of
   notebook 02 with rain set to zero, at the 2023 level), used to turn the expected % rise into cases.

Outputs: `ncr_month_outlook.csv`, `ncr_flood_events.csv`, `province_points.csv`, `weekly_case_baseline.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import statsmodels.formula.api as smf

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
RAW, OUT = ROOT / 'datasets', ROOT / 'data' / 'processed'
NCR = 'PH13074'

r = pd.read_csv(RAW / 'phl-rainfall-subnat-full.csv', usecols=['date', 'adm_level', 'PCODE', 'rfh'])
r = r[(r['adm_level'] == 2) & (r['PCODE'] == NCR)].copy()
r['date'] = pd.to_datetime(r['date'])
r['year'], r['month'] = r['date'].dt.year, r['date'].dt.month
ALERT = pd.read_csv(OUT / 'dekad_rain.csv').query('PCODE == @NCR')['p90_mm'].iloc[0]
print(len(r), 'NCR 10-day periods,', r['date'].min().date(), 'to', r['date'].max().date(), '| alert threshold', ALERT, 'mm')

1649 NCR 10-day periods, 1981-01-01 to 2026-10-11 | alert threshold 166.39 mm


## Flood level from known floods

Each event is matched to the 10-day period that holds its peak rain day (CHIRPS periods start on the 1st, 11th and
21st of each month).

In [2]:
EVENTS = {'Ondoy (Ketsana), 26 Sep 2009': '2009-09-21', 'Habagat, 7 Aug 2012': '2012-08-01',
          'Maring (Trami), 19 Aug 2013': '2013-08-11', 'Habagat, 11 Aug 2018': '2018-08-11',
          'Carina (Gaemi), 24 Jul 2024': '2024-07-21'}
ev = pd.DataFrame([{'event': k, 'date': pd.Timestamp(d), 'rfh': r.loc[r['date'] == d, 'rfh'].iloc[0]}
                   for k, d in EVENTS.items()])
ev['rfh'] = ev['rfh'].round(0)
ev['percentile'] = ev['rfh'].apply(lambda x: round((r['rfh'] < x).mean() * 100, 1))
FLOOD = float(ev['rfh'].min() // 10 * 10)   # round down to 10 mm
print('Flood level:', FLOOD, 'mm in 10 days; reached in', round((r['rfh'] >= FLOOD).mean() * 100, 1), '% of all periods')
ev

Flood level: 250.0 mm in 10 days; reached in 3.8 % of all periods


,event,date,rfh,percentile
0,"Ondoy (Ketsana), 26 Sep 2009",2009-09-21,313.0,98.9
1,"Habagat, 7 Aug 2012",2012-08-01,637.0,99.9
2,"Maring (Trami), 19 Aug 2013",2013-08-11,308.0,98.8
3,"Habagat, 11 Aug 2018",2018-08-11,259.0,96.8
4,"Carina (Gaemi), 24 Jul 2024",2024-07-21,280.0,97.9


## Monthly odds

Share of years (1981 to 2025) in which the month has at least one 10-day period above each level, for the whole
record and for its two halves, to see whether extremes have become more common.

In [3]:
full = r[r['year'] <= 2025]
def odds(d, level):
    return d.assign(a=d['rfh'] > level).groupby(['year', 'month'])['a'].any().groupby('month').mean()
out = pd.DataFrame({
    'mean_mm_month': full.groupby(['year', 'month'])['rfh'].sum().groupby('month').mean().round(0),
    'chance_alert': odds(full, ALERT), 'chance_flood': odds(full, FLOOD),
    'chance_alert_1981_2003': odds(full[full['year'] <= 2003], ALERT),
    'chance_alert_2004_2025': odds(full[full['year'] >= 2004], ALERT),
}).reset_index()
out['alert_mm'], out['flood_mm'] = ALERT, FLOOD
out.round(2)

,month,mean_mm_month,chance_alert,chance_flood,chance_alert_1981_2003,chance_alert_2004_2025,alert_mm,flood_mm
0,1,24.0,0.00,0.00,0.00,0.00,166.39,250.0
1,2,13.0,0.00,0.00,0.00,0.00,166.39,250.0
2,3,20.0,0.00,0.00,0.00,0.00,166.39,250.0
3,4,26.0,0.00,0.00,0.00,0.00,166.39,250.0
4,5,157.0,0.09,0.04,0.13,0.05,166.39,250.0
5,6,281.0,0.29,0.09,0.26,0.32,166.39,250.0
6,7,447.0,0.69,0.36,0.70,0.68,166.39,250.0
7,8,445.0,0.64,0.27,0.43,0.86,166.39,250.0
8,9,404.0,0.67,0.31,0.57,0.77,166.39,250.0
9,10,247.0,0.22,0.07,0.26,0.18,166.39,250.0


In [4]:
yr = full.assign(a=full['rfh'] > ALERT).groupby('year')['a'].sum()
print('Alert periods per year, 1981-2003:', round(yr[yr.index <= 2003].mean(), 1),
      '| 2004-2025:', round(yr[yr.index >= 2004].mean(), 1))
print('Years with at least one flood-level period:', int((full.groupby('year')['rfh'].max() >= FLOOD).sum()), 'of', full['year'].nunique())

Alert periods per year, 1981-2003: 3.2 | 2004-2025: 4.1
Years with at least one flood-level period: 32 of 45


## Inputs for the live forecast

In [5]:
prov = gpd.read_file(OUT / 'provinces.geojson')
pt = prov.to_crs(32651).representative_point().to_crs(4326)
points = pd.DataFrame({'PCODE': prov['PCODE'], 'lat': pt.y.round(4), 'lon': pt.x.round(4)})

w = pd.read_csv(OUT / 'weekly_national.csv').sort_values(['iso_year', 'iso_week']).reset_index(drop=True)
FC_LAGS = range(2, 7)
for L in FC_LAGS:
    w[f'share_l{L}'] = w['share_prov_above_p90'].shift(L) * 10
for k in (1, 2):
    w[f's{k}'] = np.sin(2 * np.pi * k * w['iso_week'] / 52)
    w[f'c{k}'] = np.cos(2 * np.pi * k * w['iso_week'] / 52)
d = w.dropna(subset=['cases'])
RAIN = ' + '.join(f'share_l{L}' for L in FC_LAGS)
m = smf.negativebinomial(f'cases ~ s1 + c1 + s2 + c2 + C(iso_year) + {RAIN}', d).fit(disp=0, maxiter=500)
X = pd.DataFrame({'iso_week': range(1, 53), 'iso_year': 2023, **{f'share_l{L}': 0.0 for L in FC_LAGS}})
for k in (1, 2):
    X[f's{k}'] = np.sin(2 * np.pi * k * X['iso_week'] / 52)
    X[f'c{k}'] = np.cos(2 * np.pi * k * X['iso_week'] / 52)
base = X[['iso_week']].assign(baseline_cases=m.predict(X).round(1).values)
print(len(points), 'province points; baseline from', len(d), 'weeks of cases:',
      base['baseline_cases'].min(), 'to', base['baseline_cases'].max(), 'cases a week')

81 province points; baseline from 227 weeks of cases: 43.6 to 130.2 cases a week


In [6]:
out.round(3).to_csv(OUT / 'ncr_month_outlook.csv', index=False)
ev.assign(date=ev['date'].dt.date).to_csv(OUT / 'ncr_flood_events.csv', index=False)
points.to_csv(OUT / 'province_points.csv', index=False)
base.to_csv(OUT / 'weekly_case_baseline.csv', index=False)
print('saved')

saved
